# 02 - Discover USB candidates

Enumerate only USB descriptors for the project's VID/PID. No interface claim, UART setup,
APT request, enable, or motion command is issued. Descriptor reads can still fail if the
driver is unsuitable or another application owns the device.

Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


In [ ]:
RUN_DISCOVERY = False  # Set True to enumerate physical USB devices.

## Enumerate and release descriptor handles

A matching USB ID is only a candidate. Record product and serial, then compare the serial
with the label on the intended mount. An unreadable descriptor is reported as an error,
not treated as evidence that the mount is absent. Connection in notebook 03 verifies
APT identity against the supplied serial and the supported model names.


In [ ]:
import libusb_package
import usb.core
import usb.util


def discover_candidates():
    backend = libusb_package.get_libusb1_backend()
    if backend is None:
        raise RuntimeError("No libusb backend; complete notebook 01 first.")
    rows = []
    devices = usb.core.find(
        find_all=True,
        idVendor=0x0403,
        idProduct=0xFAF0,
        backend=backend,
    )
    for device in devices:
        row = {
            "vid": f"{device.idVendor:04X}",
            "pid": f"{device.idProduct:04X}",
            "bcdDevice": f"{device.bcdDevice:04X}",
        }
        try:
            for name in ("manufacturer", "product", "serial_number"):
                try:
                    row[name] = getattr(device, name)
                except (usb.core.USBError, ValueError) as error:
                    row[name] = None
                    row[f"{name}_error"] = f"{type(error).__name__}: {error}"
        finally:
            usb.util.dispose_resources(device)
        rows.append(row)
    return rows


if RUN_DISCOVERY:
    candidates = discover_candidates()
    for candidate in candidates:
        print(candidate)
    if not candidates:
        print("No USB candidates returned. Check notebook 01, power, cable, and ownership.")
else:
    print("Discovery skipped; set RUN_DISCOVERY = True to enumerate devices.")

The current transport accepts bridge revision `bcdDevice=1000`, one interface, and two
64-byte bulk endpoints. Discovery does not prove that all transport checks will pass.
Multiple candidates are allowed here; nothing selects the first device automatically.

Next: [03 - Connection and identity](03_connection.ipynb).
